In [18]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist   # use tensorflow to load in dataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import torch.nn as nn
import torch.optim as optim
import torch

# Convolutional Neural Networks
Convolutional Neural Networks (CNNs) are a class of deep learning models designed for data with spatial structure, such as images, videos, and spectrograms of audio. Unlike a traditional fully connected neural network, a CNN learns local patterns by applying small learnable filters, called kernels, across the input. These filters automatically learn useful features such as edges, textures, shapes, and eventually entire objects as the network becomes deeper.

## What is a Convolution?
A convolution applies a small ($k \times k$) matrix, called a kernel (or filter), across an input image. At each location, the kernel and the corresponding image patch are multiplied element-wise and summed to produce a single output value. This operation is essentially a sliding dot product.

The resulting values form a feature map, which highlights where the learned feature appears in the image.

The movement of the kernel is controlled by the stride, which specifies how many pixels the kernel moves after each computation. A stride of 1 examines every possible location, while larger strides reduce the output size and perform downsampling.

For an input image of size ($H \times W$), a kernel of size ($K \times K$), stride ($S$), and no padding, the output dimensions are

$$
H_{out} = \frac{H-K}{S}+1,\qquad
W_{out} = \frac{W-K}{S}+1.
$$

## Pooling Layer
Pooling layers reduce the spatial dimensions of the feature maps while retaining the most important information. This decreases memory usage, reduces computation, and helps make the network more robust to small translations of the input.

The most common pooling operation is max pooling. A small window (typically ($2 \times 2$)) slides across the feature map, and only the maximum value within each window is kept.

For example,

$$
\begin{bmatrix}
2 & 5\\
1 & 4
\end{bmatrix}
\rightarrow 5
$$

Because only the strongest activation is preserved, max pooling emphasizes the most prominent detected features.

## Softmax Layer
The final layer of a classification CNN is usually a fully connected layer followed by a softmax activation.

The fully connected layer combines all extracted features into a vector of class scores (called logits). The softmax function converts these scores into probabilities that sum to one:

$$
P(y=i)=\frac{e^{z_i}}{\sum_j e^{z_j}},
$$

where $z_i$ is the score for class $i$.

The predicted class is simply the one with the highest probability.

## LeNET
The example implemented in this chapter is LeNet, one of the simplest convolutional neural networks. Its architecture consists of:

1. Convolution layer
2. Max pooling layer
3. Fully connected layer with a softmax output

The convolution layer detects local image features, the pooling layer reduces the feature map size, and the final layer performs classification.

Modern CNNs repeat the convolution and pooling stages multiple times before reaching the final classification layer, allowing them to learn increasingly complex visual features. Below is a from scratch implementation

In [26]:
def softmax(A):
    expA = np.exp(A)
    return expA / expA.sum(axis=0, keepdims=True)

def ReLU(A):
    return [max(0,i) for i in A]

def one_hot(y):
    y = pd.Series(y)
    y = pd.get_dummies(y).values.tolist()
    return np.array(y)

class ConvolutionLayer:
    def __init__(self, kernel_num, kernel_size):
        self.kernel_num = kernel_num
        self.kernel_size = kernel_size
        # make random filters to start
        self.kernels = np.random.randn(kernel_num, kernel_size, kernel_size) / (kernel_size**2)

    def patches_generator(self, image):
        # Extract image height and width
        hei, wei = image.shape
        self.image = image
        # The number of patches, given a fxf filter is h-f+1 for height and w-f+1 for width
        for h in range(hei-self.kernel_size+1):
            for w in range(wei-self.kernel_size+1):
                patch = image[h:(h+self.kernel_size), w:(w+self.kernel_size)]
                yield patch, h, w   # yield is return but for iteration
    
    def forward_prop(self, image):
        hei, wei = image.shape
        # Initialize the convolution output volume of the correct size
        convolution_output = np.zeros((hei-self.kernel_size+1, wei-self.kernel_size+1, self.kernel_num))
        # Unpack the generator
        for patch, h, w in self.patches_generator(image):
            # Perform convolution for each patch
            convolution_output[h,w] = np.sum(patch*self.kernels, axis=(1,2))
        return convolution_output
    
    def back_prop(self, dE_dY, alpha):
        # Initialize gradient of the loss function with respect to the kernel weights
        dE_dk = np.zeros(self.kernels.shape)
        for patch, h, w in self.patches_generator(self.image):
            for f in range(self.kernel_num):
                dE_dk[f] += patch * dE_dY[h, w, f]
        # Update the parameters
        self.kernels -= alpha*dE_dk
        return dE_dk

class MaxPoolingLayer:
    def __init__(self, kernel_size):
        self.kernel_size = kernel_size

    def patches_generator(self, image):
        # Compute the ouput size
        output_h = int(image.shape[0]/self.kernel_size)
        output_w = int(image.shape[1]/self.kernel_size)
        self.image = image

        for h in range(output_h):
            for w in range(output_w):
                patch = image[(h*self.kernel_size):(h*self.kernel_size+self.kernel_size), (w*self.kernel_size):(w*self.kernel_size+self.kernel_size)]
                yield patch, h, w

    def forward_prop(self, image):
        image_h, image_w, num_kernels = image.shape
        max_pooling_output = np.zeros((int(image_h/self.kernel_size), int(image_w/self.kernel_size), num_kernels))
        for patch, h, w in self.patches_generator(image):
            max_pooling_output[h,w] = np.amax(patch, axis=(0,1))
        return max_pooling_output

    def back_prop(self, dE_dY):
        dE_dk = np.zeros(self.image.shape)
        for patch,h,w in self.patches_generator(self.image):
            image_h, image_w, num_kernels = patch.shape
            max_val = np.amax(patch, axis=(0,1))

            for idx_h in range(image_h):
                for idx_w in range(image_w):
                    for idx_k in range(num_kernels):
                        if patch[idx_h,idx_w,idx_k] == max_val[idx_k]:
                            dE_dk[h*self.kernel_size+idx_h, w*self.kernel_size+idx_w, idx_k] = dE_dY[h,w,idx_k]
            return dE_dk

class SoftmaxLayer:
    def __init__(self, input_units, output_units):
        # Initiallize weights and biases
        self.weight = np.random.randn(input_units, output_units)/input_units
        self.bias = np.zeros(output_units)

    def forward_prop(self, image):
        self.original_shape = image.shape # stored for backprop
        # Flatten the image
        image_flattened = image.flatten()
        self.flattened_input = image_flattened # stored for backprop
        # Perform matrix multiplication and add bias
        first_output = np.dot(image_flattened, self.weight) + self.bias
        self.output = first_output
        # Apply softmax activation
        softmax_output = softmax(first_output)
        return softmax_output

    def back_prop(self, dE_dY, alpha):
        for i, gradient in enumerate(dE_dY):
            if gradient == 0:
                continue
            transformation_eq = np.exp(self.output)
            S_total = np.sum(transformation_eq)

            # Compute gradients with respect to output (Z)
            dY_dZ = -transformation_eq[i]*transformation_eq / (S_total**2)
            dY_dZ[i] = transformation_eq[i]*(S_total - transformation_eq[i]) / (S_total**2)

            # Compute gradients of output Z with respect to weight, bias, input
            dZ_dw = self.flattened_input
            dZ_db = 1
            dZ_dX = self.weight

            # Gradient of loss with respect ot output
            dE_dZ = gradient * dY_dZ

            # Gradient of loss with respect to weight, bias, input
            dE_dw = dZ_dw[np.newaxis].T @ dE_dZ[np.newaxis]
            dE_db = dE_dZ * dZ_db
            dE_dX = dZ_dX @ dE_dZ

            # Update parameters
            self.weight -= alpha*dE_dw
            self.bias -= alpha*dE_db

            return dE_dX.reshape(self.original_shape)

class CNN:
    def __init__(self, layers, lr = 0.05):
        self.layers = layers
        self.learning_rate = lr

    def forward(self, image, label=None):
        output = image/255.
        for layer in self.layers:
            output = layer.forward_prop(output)

        return output

    def compute_loss(self, output, label):
        return -np.log(output[label] + 1e-12)

    def backward(self, gradient):
        grad_back = gradient
        for layer in self.layers[::-1]:
            if type(layer) in [ConvolutionLayer, SoftmaxLayer]:
                grad_back = layer.back_prop(grad_back, self.learning_rate)
            elif type(layer) == MaxPoolingLayer:
                grad_back = layer.back_prop(grad_back)
        return grad_back

    def train_step(self, image, label):
        # Forward pass
        output = self.forward(image, label)
        loss = self.compute_loss(output, label)

        # Initial gradient
        gradient = np.zeros_like(output)
        gradient[label] = -1 / (output[label] + 1e-12)

        # Backward pass
        self.backward(gradient)

        prediction = np.argmax(output)

        return prediction, loss

    def predict(self, image):
        output = self.forward(image)
        return np.argmax(output)

    def predict_proba(self, image):
        return self.forward(image)

    def evaluate(self, X, y):
        preds = []

        for image in X:
            preds.append(self.predict(image))

        preds = np.array(preds)
        accuracy = np.mean(preds == y) * 100

        return accuracy

In [27]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)

train_images = train_images[:2000]
train_labels = train_labels[:2000]

test_images = test_images[:500]
test_labels = test_labels[:500]

(60000, 28, 28)
(60000,)
(10000, 28, 28)


In [28]:
# this code will take a while to run

model = CNN(
    layers=[
        ConvolutionLayer(kernel_num=8, kernel_size=3),
        MaxPoolingLayer(kernel_size=2),
        SoftmaxLayer(13 * 13 * 8, 10)
    ],
    lr=0.05
)

epochs = 5
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    for image, label in zip(train_images, train_labels):
        pred, loss = model.train_step(image, label)
        total_loss += loss
        if pred == label:
            correct += 1

    print(f"Epoch {epoch+1}: " f"Loss = {total_loss / len(train_images):.4f}, " f"Accuracy = {100 * correct / len(train_images):.2f}%")

Epoch 1: Loss = 0.7804, Accuracy = 80.90%
Epoch 2: Loss = 0.3922, Accuracy = 90.20%
Epoch 3: Loss = 0.3142, Accuracy = 92.00%
Epoch 4: Loss = 0.2699, Accuracy = 93.35%
Epoch 5: Loss = 0.2392, Accuracy = 94.15%


In [29]:
print(type(test_images), test_images.shape, test_images.dtype)
print(type(test_images[0]), test_images[0].shape)

out = model.forward(test_images[0])
print(type(out), out.shape if hasattr(out, "shape") else out)

<class 'numpy.ndarray'> (500, 28, 28) uint8
<class 'numpy.ndarray'> (28, 28)
<class 'numpy.ndarray'> (10,)


In [30]:
# testing and metrics

y_preds = np.array([model.predict(image) for image in test_images])

# Accuracy
accuracy = accuracy_score(test_labels, y_preds)
print(f"Model Accuracy: {accuracy:.2%}")

# Confusion matrix
cm = confusion_matrix(test_labels, y_preds)
print("Confusion Matrix:")
print(cm)

# Precision, recall, F1-score
print("\nClassification Report:")
print(classification_report(test_labels, y_preds))

Model Accuracy: 90.00%
Confusion Matrix:
[[41  0  0  1  0  0  0  0  0  0]
 [ 0 67  0  0  0  0  0  0  0  0]
 [ 0  0 50  0  0  0  0  3  1  1]
 [ 0  0  3 38  0  3  0  1  0  0]
 [ 1  0  0  0 50  0  2  0  0  2]
 [ 1  0  2  1  0 45  0  1  0  0]
 [ 1  0  2  0  1  2 37  0  0  0]
 [ 0  0  3  1  1  0  0 43  0  1]
 [ 1  0  0  1  1  1  0  2 33  1]
 [ 0  0  0  3  0  1  0  3  1 46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.98      0.94        42
           1       1.00      1.00      1.00        67
           2       0.83      0.91      0.87        55
           3       0.84      0.84      0.84        45
           4       0.94      0.91      0.93        55
           5       0.87      0.90      0.88        50
           6       0.95      0.86      0.90        43
           7       0.81      0.88      0.84        49
           8       0.94      0.82      0.88        40
           9       0.90      0.85      0.88        54

    accu